# 07 — Menambah Fitur NDVI dari Google Earth Engine

apakah indeks vegetasi (NDVI) proxy tutupan hijau meningkatkan prediksi PM2.5?

source: `MODIS/061/MOD13A2` (NDVI 16-hari, 1 km) di GEE, di-resample harian via interpolasi linear.

notes:

cara setup untuk gee
1. Daftar di https://earthengine.google.com/ → autentikasi.
2. Buat GCP project di https://console.cloud.google.com/ - catat **Project ID**.
3. Run cell `pip install` di bawah jika belum.
4. Edit `PROJECT_ID` di config cell, lalu run sel autentikasi.

Cell ekstraksi sudah punya **caching** — kalau `ndvi_<station>.csv` sudah ada, stasiun tersebut akan di-skip

## A. Setup library & autentikasi

In [1]:
import sys
!{sys.executable} -m pip install -q earthengine-api


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import sys, pathlib, time
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from src import config as C

PROJECT_ID = 'pm25-prediction-494706'  
START = '2022-01-01'
END   = '2024-12-31'
BUFFER_M = 2750  # ~5,5 km diameter 
FORCE_REFETCH = False  # True kalau mau download ulang meskipun cache ada

In [ ]:
# Autentikasi & init GEE.
# notes kalau ee.Initialize gagal, jangan jalankan ee.Authenticate() di notebook
# karena VS Code/Jupyter sering tidak render input prompt-nya.
# Solusi: buka terminal Windows (CMD/PowerShell), jalankan:
# earthengine authenticate

import ee

ee.Initialize(project=PROJECT_ID)
print('GEE init OK')


GEE init OK


## B. Ekstraksi NDVI per stasiun (robust + caching)

Perbaikan vs versi lama:
1. **Single `getInfo()`** - ambil date+NDVI sekaligus dalam satu fitur, pasangan terjamin.
2. **Try/except per stasiun** - kalau 1 stasiun gagal (timeout/quota), stasiun lain tetap lanjut.
3. **Skip-if-exists** - kalau file `ndvi_<station>.csv` sudah ada, lewati (kecuali `FORCE_REFETCH=True`).
4. **Retry sederhana** - coba ulang sampai 3x dengan backoff bertingkat untuk error transient (network/quota).

In [ ]:
def fetch_ndvi_for_station(station: str, lat: float, lon: float, max_retries: int = 3) -> pd.DataFrame:
    """Ekstrak NDVI MOD13A2 untuk satu titik stasiun (buffer ~5,5 km)."""
    pt = ee.Geometry.Point([lon, lat]).buffer(BUFFER_M)

    def reduce_img(img):
        # Reduksi rerata NDVI dalam buffer
        mean = img.reduceRegion(ee.Reducer.mean(), pt, 1000).get('NDVI')
        return ee.Feature(None, {
            'date': img.date().format('YYYY-MM-dd'),
            'NDVI': mean,
        })

    coll = (
        ee.ImageCollection('MODIS/061/MOD13A2')
          .filterDate(START, END)
          .select('NDVI')
    )
    feats = coll.map(reduce_img).filter(ee.Filter.notNull(['NDVI']))

    # SINGLE getInfo, ambil semua atribut sekaligus, pairing terjamin
    last_err = None
    payload = None
    for attempt in range(1, max_retries + 1):
        try:
            payload = feats.getInfo()
            break
        except Exception as e:
            last_err = e
            wait = 5 * (attempt ** 2)  # 5s, 20s, 45s
            print(f'    ! attempt {attempt} gagal ({type(e).__name__}: {e}). Retry dalam {wait}s')
            time.sleep(wait)
    if payload is None:
        raise RuntimeError(f'gagal {max_retries}x: {last_err}')

    rows = [
        {'date': f['properties']['date'], 'NDVI_raw': f['properties']['NDVI']}
        for f in payload['features']
    ]
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f'NDVI kosong untuk {station} - periksa koordinat')

    df['date'] = pd.to_datetime(df['date'])
    df['NDVI'] = df['NDVI_raw'] * 0.0001  # MOD13A2 scale factor
    df = df.sort_values('date').drop(columns='NDVI_raw').reset_index(drop=True)

    # Resample harian (NDVI berubah lambat, interpolasi linear OK)
    full = pd.date_range(START, END, freq='D')
    df_daily = (
        df.set_index('date')
          .reindex(full)
          .interpolate('linear')
          .ffill().bfill()
    )
    df_daily.index.name = 'date'
    return df_daily.reset_index()

In [ ]:
# Loop semua stasiun (gagal di 1 stasiun TIDAK menghentikan stasiun lain).
results_log = []
for s, meta in C.STATIONS.items():
    out_path = C.EXTERNAL_DIR / f'ndvi_{s}.csv'

    if out_path.exists() and not FORCE_REFETCH:
        print(f'[SKIP] {s} - cache ada di {out_path.name}')
        results_log.append({'station': s, 'status': 'cached'})
        continue

    print(f'[FETCH] {s} (lat={meta["lat"]}, lon={meta["lon"]}) ...')
    try:
        t0 = time.time()
        df_daily = fetch_ndvi_for_station(s, meta['lat'], meta['lon'])
        df_daily.to_csv(out_path, index=False)
        elapsed = time.time() - t0
        print(f'   OK  {len(df_daily)} hari NDVI tersimpan ({elapsed:.1f}s)')
        results_log.append({'station': s, 'status': 'ok', 'rows': len(df_daily), 'sec': round(elapsed, 1)})
    except Exception as e:
        print(f'   GAGAL: {type(e).__name__}: {e}')
        results_log.append({'station': s, 'status': 'failed', 'error': str(e)})

pd.DataFrame(results_log)

[FETCH] us_embassy_1 (lat=-6.1811, lon=106.8279) ...
   OK  1096 hari NDVI tersimpan (1.1s)
[FETCH] us_embassy_2 (lat=-6.2366, lon=106.7931) ...
   OK  1096 hari NDVI tersimpan (0.9s)
[FETCH] jakarta_gbk (lat=-6.2155, lon=106.803) ...
   OK  1096 hari NDVI tersimpan (0.9s)
[FETCH] bundaran_hi (lat=-6.1946, lon=106.8235) ...
   OK  1096 hari NDVI tersimpan (0.8s)
[FETCH] kelapa_gading (lat=-6.1535, lon=106.9108) ...
   OK  1096 hari NDVI tersimpan (1.0s)
[FETCH] jagakarsa (lat=-6.3569, lon=106.8036) ...
   OK  1096 hari NDVI tersimpan (0.9s)
[FETCH] lubang_buaya (lat=-6.2888, lon=106.9091) ...
   OK  1096 hari NDVI tersimpan (0.8s)
[FETCH] kebun_jeruk (lat=-6.2073, lon=106.7531) ...
   OK  1096 hari NDVI tersimpan (0.9s)


,station,status,rows,sec
0,us_embassy_1,ok,1096,1.1
1,us_embassy_2,ok,1096,0.9
2,jakarta_gbk,ok,1096,0.9
3,bundaran_hi,ok,1096,0.8
4,kelapa_gading,ok,1096,1.0
5,jagakarsa,ok,1096,0.9
6,lubang_buaya,ok,1096,0.8
7,kebun_jeruk,ok,1096,0.9


In [7]:
# Sanity check: lihat sample dari salah satu stasiun
sample_path = C.EXTERNAL_DIR / 'ndvi_kelapa_gading.csv'
if sample_path.exists():
    sample = pd.read_csv(sample_path, parse_dates=['date'])
    print(sample.head())
    print(f'\nStatistik NDVI: min={sample["NDVI"].min():.3f} max={sample["NDVI"].max():.3f} mean={sample["NDVI"].mean():.3f}')

        date      NDVI
0 2022-01-01  0.251219
1 2022-01-02  0.246514
2 2022-01-03  0.241810
3 2022-01-04  0.237105
4 2022-01-05  0.232401

Statistik NDVI: min=0.077 max=0.306 mean=0.227


## C. Gabung NDVI ke dataset, latih ulang model, bandingkan

In [8]:
from src.data_loader import load_all_stations
from src.evaluation import compute_metrics
from src.model import build_lstm, set_seed, train_model
from src.preprocessing import build_pipeline, inverse_target

FIXED_PARAMS = {'lstm_units': 32, 'dropout_rate': 0.1, 'optimizer': 'adam', 'learning_rate': 1e-3}
EPOCHS = 100
PATIENCE = 15

BASE      = C.WEATHER_COLS + [C.AOD_COL, C.TARGET]
WITH_NDVI = C.WEATHER_COLS + [C.AOD_COL, 'NDVI', C.TARGET]

dfs = load_all_stations(reindex=True)

In [9]:
def merge_ndvi(df, station):
    path = C.EXTERNAL_DIR / f'ndvi_{station}.csv'
    if not path.exists():
        return None
    ndvi = pd.read_csv(path, parse_dates=['date']).rename(columns={'date': C.DATE_COL})
    return df.merge(ndvi, on=C.DATE_COL, how='left')


def run_with_features(df, features):
    data = build_pipeline(df, features, smooth_aod=False)
    set_seed()
    n_features = data['X_train'].shape[2]
    model = build_lstm(input_shape=(C.LOOKBACK, n_features), **FIXED_PARAMS)
    train_model(model, data['X_train'], data['y_train'], data['X_val'], data['y_val'],
                epochs=EPOCHS, patience=PATIENCE, verbose=0)
    y_pred = model.predict(data['X_test'], verbose=0).flatten()
    y_pred = inverse_target(y_pred, data['scaler'], data['target_idx'], n_features)
    y_true = inverse_target(data['y_test'], data['scaler'], data['target_idx'], n_features)
    return compute_metrics(y_true, y_pred)

In [10]:
rows = []
for s in C.HEALTHY_STATIONS:
    df = dfs[s]
    df_ndvi = merge_ndvi(df, s)
    if df_ndvi is None:
        print(f'  ! NDVI untuk {s} belum ada, lewati')
        continue

    m_base = run_with_features(df, BASE)
    m_ndvi = run_with_features(df_ndvi, WITH_NDVI)

    rows.append({
        'station': s,
        'R2_base':  m_base['R2'],  'RMSE_base':  m_base['RMSE'],  'MAE_base':  m_base['MAE'],
        'R2_ndvi':  m_ndvi['R2'],  'RMSE_ndvi':  m_ndvi['RMSE'],  'MAE_ndvi':  m_ndvi['MAE'],
    })
    print(f'  {s} OK   delta_R2 = {m_ndvi["R2"] - m_base["R2"]:+.4f}')

df_ndvi_cmp = pd.DataFrame(rows)
df_ndvi_cmp['delta_R2'] = df_ndvi_cmp['R2_ndvi'] - df_ndvi_cmp['R2_base']
df_ndvi_cmp.to_csv(C.METRICS_DIR / '07_ndvi_comparison.csv', index=False)
df_ndvi_cmp

  us_embassy_1 OK   delta_R2 = +4682.4116
  us_embassy_2 OK   delta_R2 = +0.0937
  bundaran_hi OK   delta_R2 = -0.0424
  kelapa_gading OK   delta_R2 = +0.1605
  jagakarsa OK   delta_R2 = +0.0193
  lubang_buaya OK   delta_R2 = +0.1937
  kebun_jeruk OK   delta_R2 = +0.0964


,station,R2_base,RMSE_base,MAE_base,R2_ndvi,RMSE_ndvi,MAE_ndvi,delta_R2
0,us_embassy_1,-5973.522653,1629.921874,523.162729,-1291.111076,757.992846,356.647847,4682.411577
1,us_embassy_2,0.421863,19.275787,14.289577,0.515602,17.644042,12.579689,0.093739
2,bundaran_hi,0.517305,20.861740,15.131587,0.474913,21.758536,15.311424,-0.042392
3,kelapa_gading,0.613574,14.309174,11.063613,0.774067,10.941363,8.109326,0.160493
4,jagakarsa,0.489004,13.167948,8.872371,0.508261,12.917448,8.612594,0.019257
5,lubang_buaya,-0.232382,27.330646,21.489693,-0.038669,25.090877,20.949075,0.193713
6,kebun_jeruk,0.551302,20.666875,15.799685,0.647748,18.311541,13.983899,0.096445
